In [ ]:
%config InlineBackend.figure_format = 'retina'

import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

from dotenv import load_dotenv
from matplotlib.patches import Patch
from openinference.semconv.trace import (
    OpenInferenceSpanKindValues,
    SpanAttributes
)
from sklearn.metrics import precision_score, recall_score, roc_auc_score

load_dotenv()

SPAN_KIND = SpanAttributes.OPENINFERENCE_SPAN_KIND
OUTPUT_VALUE = SpanAttributes.OUTPUT_VALUE
AGENT = OpenInferenceSpanKindValues.AGENT.value

In [ ]:
# original_input_path = "../data/frames/llm_frames_results_all_ctx_40960_kvq_f16.csv"
original_input_path = "../data/frames/llm_frames_results_judged.csv"
original_input = pd.read_csv(original_input_path)
questions = original_input["Prompt"]
answers = original_input["Answer"]

base_path = "../logs/frames_full_llamacpp_qwen3_30b"
output_file = base_path.split("/")[-1]
max_folder = max(int(f) for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)))
data = []

for i in range(max_folder + 1):
    path = f"{base_path}/{i}"
    for run_idx in range(1):
        run_path = f"{path}/run_{run_idx}"
        summary_path = f"{run_path}/analysis/summary.json"
        step_summary_path = f"{run_path}/analysis/step_summary.json"

        with open(summary_path, "r") as f:
            summary = json.load(f)
        with open(step_summary_path, "r") as f:
            step_summary = json.load(f)

        step_data = {}
        step_idx = 0
        for step in step_summary:
            if step["kind"] != "LLM":
                continue
            for k, v in step.items():
                step_data[f"step_{step_idx}_{k}"] = v
            step_idx += 1

        agent_output = summary["agent_output"]
        if agent_output is not None and "<think>" in agent_output and "</think>" in agent_output:
            summary["agent_output"] = agent_output[agent_output.find("</think>")+len("</think>"):].strip()

        data.append({
            "question": questions[i],
            "answer": answers[i],
            **summary,
            **step_data,
        })

data = pd.DataFrame.from_dict(data)
data.to_csv(f"../data/frames/profile_results_{output_file}.csv", index=False)

In [ ]:
# SimpleQA
original_input_path = "../data/simpleqa/llm_simpleqa_results_qwen3_30b_2507_judged.csv"
original_input = pd.read_csv(original_input_path)
questions = original_input["problem"]
answers = original_input["answer"]

base_path = "../logs/simpleqa_full_llamacpp_qwen3_30b"
output_file = base_path.split("/")[-1]
max_folder = max(int(f) for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)))
data = []

for i in range(max_folder + 1):
    path = f"{base_path}/{i}"
    for run_idx in range(1):
        run_path = f"{path}/run_{run_idx}"
        summary_path = f"{run_path}/analysis/summary.json"
        step_summary_path = f"{run_path}/analysis/step_summary.json"

        with open(summary_path, "r") as f:
            summary = json.load(f)
        with open(step_summary_path, "r") as f:
            step_summary = json.load(f)

        step_data = {}
        step_idx = 0
        for step in step_summary:
            if step["kind"] != "LLM":
                continue
            for k, v in step.items():
                step_data[f"step_{step_idx}_{k}"] = v
            step_idx += 1

        agent_output = summary["agent_output"]
        if agent_output is not None and "<think>" in agent_output and "</think>" in agent_output:
            summary["agent_output"] = agent_output[agent_output.find("</think>")+len("</think>"):].strip()

        data.append({
            "question": questions[i],
            "answer": answers[i],
            **summary,
            **step_data,
        })

data = pd.DataFrame.from_dict(data)
data.to_csv(f"../data/simpleqa/profile_results_{output_file}.csv", index=False)

In [ ]:
base_path = "../logs/frames_full_llamacpp_qwen3_30b"
max_folder = max(int(f) for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)))
failed_runs = []

for i in range(max_folder + 1):
    path = f"{base_path}/{i}"
    for run_idx in range(1):
        run_path = f"{path}/run_{run_idx}"
        summary_path = f"{run_path}/analysis/summary.json"
        step_summary_path = f"{run_path}/analysis/step_summary.json"

        try:
            with open(summary_path, "r") as f:
                summary = json.load(f)
            with open(step_summary_path, "r") as f:
                step_summary = json.load(f)
        except:
            failed_runs.append(i)
            break

        if summary["duration_sec"] > 300:
            failed_runs.append(i)
            break

print(failed_runs)

In [ ]:
acc = []
energy_usage = []
labels = []

inputs = {
    # "Qwen 3 1.7B": "../data/frames/frames_profile_results_qwen3_1.7b_judged.csv",
    "Qwen 3 1.7B LlamaCpp": "../data/frames/profile_results_frames_full_llamacpp_qwen3_1.7b_judged.csv",
    # "Qwen 3 4B": "../data/frames/frames_profile_results_qwen3_4b_judged.csv",
    # "Qwen 3 8B": "../data/frames/frames_profile_results_qwen3_8b_judged.csv",
    # "Comcade 1.7B to 30B S1": "../data/frames/profile_results_frames_ollama_qwen3_fixed_cascade_compress_1.7b_to_30b_step_1_judged.csv",
    # "Comcade 1.7B to 30B min logprobs -1.0": "../data/frames/profile_results_frames_llamacpp_min_logprobs_cascade_qwen3_1.7b_to_30b_threshold_-1.0_judged.csv",
    # "Comcade 1.7B to 30B S2": "../data/frames/profile_results_frames_ollama_qwen3_fixed_cascade_compress_1.7b_to_30b_step_2_judged.csv",
    # "Comcade 1.7B to 14B S1": "../data/frames/frames_profile_results_qwen3_fixed_cascade_compress_1.7b_to_14b_step_1_judged.csv",
    # "Comcade 1.7B to 14B min logprobs -1.0": "../data/frames/profile_results_frames_llamacpp_min_logprobs_cascade_qwen3_1.7b_to_14b_threshold_-1.0_judged.csv",
    # "Qwen 3 14B": "../data/frames/frames_profile_results_qwen3_14b_judged.csv",
    # "Qwen 3 30B A3B Instruct 2507": "../data/frames/frames_profile_results_qwen3_30b_judged.csv",
    "Qwen 3 30B LlamaCpp": "../data/frames/profile_results_frames_full_llamacpp_qwen3_30b_judged.csv",
    # "Qwen 3 32B": "../data/frames/frames_profile_results_qwen3_32b_judged.csv",
}

pairs = [
    ("Qwen 3 1.7B + 30B", "Qwen 3 1.7B LlamaCpp", "Qwen 3 30B LlamaCpp"),
    # ("Qwen 3 1.7B + 14B", "Qwen 3 1.7B", "Qwen 3 14B"),
]

sim_pos_recalls = [0.795]
sim_neg_recalls = [0.59]

for label, path in inputs.items():
    data = pd.read_csv(path)
    ac = round((data["agent_output_eval"] == "CORRECT").mean(), 2)
    acc.append(ac)
    energy_usage.append(data["energy_total_mWh"])
    labels.append(label)

    # Early exit
    for step in [4]:
        # Oracle
        exit_energy = data["step_0_energy_total_mWh"]
        for i in range(1, step):
            exit_energy = exit_energy + data[f"step_{i}_energy_total_mWh"].fillna(0)
        early_exit_energy = np.where(
            data["agent_output_eval"] == "CORRECT",
            data["energy_total_mWh"],
            exit_energy,
        )
        acc.append(ac)
        energy_usage.append(early_exit_energy)
        labels.append(f"Oracle early exit S{step} {label}")

        next_step_exit_energy = exit_energy + data[f"step_{step}_energy_total_mWh"].fillna(0)

        # Random
        for prob in [0.85, 0.9, 0.95]:
            rand_vals = np.random.rand(len(data))
            early_exit_energy = np.where(
                rand_vals <= prob,
                data["energy_total_mWh"],
                exit_energy,
            )
            ac = round(((data["agent_output_eval"] == "CORRECT") & (rand_vals <= prob)).mean(), 2)
            acc.append(ac)
            energy_usage.append(early_exit_energy)
            labels.append(f"Random exit S{step} {label} {round(1 - prob, 2)}")

        # Classifier
        classifier_prob = data[f"classifier_prob_S{step}"]
        

        # Simulation
        is_correct = data["agent_output_eval"] == "CORRECT"
        num_steps = data["num_LLM_calls"]
        for recall_pos in sim_pos_recalls:
            for recall_neg in sim_neg_recalls:
                rand_vals = np.random.rand(len(data))
                preds = np.where(data["agent_output_eval"] == "CORRECT", rand_vals <= recall_pos, rand_vals <= 1 - recall_neg)
                actual_decision = (num_steps <= step) | preds
                energy_usage.append(np.where(
                    actual_decision,
                    data["energy_total_mWh"],
                    exit_energy,
                ))
                acc.append(round((is_correct & actual_decision).mean(), 2))
                is_correct_labels = is_correct[num_steps > step]
                preds_labels = preds[num_steps > step]
                actual_recall = recall_score(is_correct_labels, preds_labels)
                actual_precision = precision_score(is_correct_labels, preds_labels)
                auc = roc_auc_score(is_correct_labels, preds_labels)
                labels.append(f"Early exit S{step} {label} {round(actual_recall, 2)} rec {round(actual_precision, 2)} prec {round(auc, 2)} auc")

                # Next step simulation
                rand_vals = np.random.rand(len(data))
                next_step_preds = np.where(data["agent_output_eval"] == "CORRECT", rand_vals <= recall_pos, rand_vals <= 1 - recall_neg)
                energy_usage.append(np.where(
                    num_steps <= step,
                    data["energy_total_mWh"],
                    np.where(
                        ~preds,
                        exit_energy,
                        np.where(
                            (num_steps <= (step + 1)) | next_step_preds,
                            data["energy_total_mWh"],
                            next_step_exit_energy,
                        ),
                    ),
                ))
                actual_decision = ((num_steps <= step) | (preds & ((num_steps <= (step+1)) | next_step_preds)))
                # acc.append(round((is_correct & actual_decision).mean(), 2))
                acc.append(round((
                    np.where(
                        num_steps <= step,
                        is_correct,
                        np.where(
                            preds & ((num_steps <= step + 1) | next_step_preds),
                            is_correct,
                            np.zeros_like(is_correct),
                        )
                    )
                ).mean(), 2))
                is_correct_labels = is_correct[num_steps > step + 1]
                preds_labels = next_step_preds[num_steps > step + 1]
                actual_recall = recall_score(is_correct_labels, preds_labels)
                actual_precision = precision_score(is_correct_labels, preds_labels)
                auc = roc_auc_score(is_correct_labels, preds_labels)
                labels.append(f"Early exit S{step}+{step+1} {label} {round(actual_recall, 2)} rec {round(actual_precision, 2)} prec {round(auc, 2)} auc")


for name, d1, d2 in pairs:
    data1 = pd.read_csv(inputs[d1])
    data2 = pd.read_csv(inputs[d2])

    # Oracle
    oracle_acc = round((
        (data1["agent_output_eval"] == "CORRECT") |
        (data2["agent_output_eval"] == "CORRECT")
    ).mean(), 2)

    oracle_energy = np.where(
        (
            (data2["agent_output_eval"] == "CORRECT") & (
                (data1["agent_output_eval"] != "CORRECT") |
                (data2["energy_total_mWh"] < data1["energy_total_mWh"])
            )
        ) | (
            (data2["agent_output_eval"] != "CORRECT") & (
                (data1["agent_output_eval"] != "CORRECT") &
                (data2["energy_total_mWh"] < data1["energy_total_mWh"])
            )
        ),
        data2["energy_total_mWh"],
        data1["energy_total_mWh"],
    )

    acc.append(oracle_acc)
    energy_usage.append(oracle_energy)
    labels.append(f"Oracle routing {name}")

    # Random routing
    for routing_prob in [0.25, 0.5, 0.75, 0.85, 0.95]:
        choices = np.random.choice(2, data1.shape[0], p=[routing_prob, 1 - routing_prob])
        acc.append(np.where(choices, data1["agent_output_eval"] == "CORRECT", data2["agent_output_eval"] == "CORRECT").mean().round(2))
        energy_usage.append(np.where(choices, data1["energy_total_mWh"], data2["energy_total_mWh"]))
        labels.append(f"Random routing {routing_prob} {name}")

    # Optimal cascade at step 2
    # optimal_cascade_energy = np.where(
    #     (data1["agent_output_eval"] == "CORRECT"),
    #     data1["energy_total_mWh"],
    #     data1["step_0_energy_total_mWh"] + data1["step_1_energy_total_mWh"].fillna(0) + data2["energy_total_mWh"],
    # )
    # acc.append(oracle_acc)
    # energy_usage.append(optimal_cascade_energy)
    # labels.append(f"Optimal cascade {name}")

    # Optimal reverse cascade at step x
    for step in [4]:
        cascade_energy = data1["energy_total_mWh"] + data2["step_0_energy_total_mWh"]
        for i in range(1, step):
            cascade_energy = cascade_energy + data2[f"step_{i}_energy_total_mWh"].fillna(0)
        next_step_cascade_energy = cascade_energy + data2[f"step_{step}_energy_total_mWh"].fillna(0)            
        is_correct = data2["agent_output_eval"] == "CORRECT"
        num_steps = data2["num_LLM_calls"]
        energy_usage.append(np.where(
            is_correct | (num_steps <= step),
            data2["energy_total_mWh"],
            cascade_energy,
        ))
        acc.append(round((
            is_correct |
            ((data1["agent_output_eval"] == "CORRECT") & (num_steps > step))
        ).mean(), 2))
        labels.append(f"Oracle reverse cascade S{step} {name}")

        # Simulated classifier
        for recall_pos in sim_pos_recalls:
            for recall_neg in sim_neg_recalls:
                rand_vals = np.random.rand(len(data2))
                preds = np.where(data2["agent_output_eval"] == "CORRECT", rand_vals <= recall_pos, rand_vals <= 1 - recall_neg)
                energy_usage.append(np.where(
                    (num_steps <= step) | preds,
                    data2["energy_total_mWh"],
                    cascade_energy,
                ))
                acc.append(round((
                    np.where(
                        (num_steps <= step) | preds,
                        is_correct,
                        data1["agent_output_eval"] == "CORRECT",
                    )
                ).mean(), 2))
                is_correct_labels = is_correct[num_steps > step]
                preds_labels = preds[num_steps > step]
                actual_recall = recall_score(is_correct_labels, preds_labels)
                actual_precision = precision_score(is_correct_labels, preds_labels)
                auc = roc_auc_score(is_correct_labels, preds_labels)
                labels.append(f"Reverse cascade S{step} {round(actual_recall, 2)} rec {round(actual_precision, 2)} prec {round(auc, 2)} auc")

                # Next step simulation
                rand_vals = np.random.rand(len(data2))
                next_step_preds = np.where(data2["agent_output_eval"] == "CORRECT", rand_vals <= recall_pos, rand_vals <= 1 - recall_neg)
                energy_usage.append(np.where(
                    num_steps <= step,
                    data2["energy_total_mWh"],
                    np.where(
                        ~preds,
                        cascade_energy,
                        np.where(
                            (num_steps <= (step + 1)) | next_step_preds,
                            data2["energy_total_mWh"],
                            next_step_cascade_energy,
                        ),
                    ),
                ))
                acc.append(round((
                    np.where(
                        (num_steps <= step) | (preds & ((num_steps <= (step + 1)) | next_step_preds)),
                        is_correct,
                        data1["agent_output_eval"] == "CORRECT",
                    )
                ).mean(), 2))
                is_correct_labels = is_correct[num_steps > step + 1]
                preds_labels = next_step_preds[num_steps > step + 1]
                actual_recall = recall_score(is_correct_labels, preds_labels)
                actual_precision = precision_score(is_correct_labels, preds_labels)
                auc = roc_auc_score(is_correct_labels, preds_labels)
                labels.append(f"Reverse cascade S{step}+{step+1} {label} {round(actual_recall, 2)} rec {round(actual_precision, 2)} prec {round(auc, 2)} auc")


    # # Optimal cascade at step 2, then early exit
    # early_exit_energy = np.where(
    #     data2["agent_output_eval"] == "CORRECT",
    #     data2["energy_total_mWh"],
    #     data2["step_0_energy_total_mWh"] + data2["step_1_energy_total_mWh"].fillna(0),
    # )
    # optimal_cascade_exit_energy = np.where(
    #     (data1["agent_output_eval"] == "CORRECT"),
    #     data1["energy_total_mWh"],
    #     data1["step_0_energy_total_mWh"] + data1["step_1_energy_total_mWh"].fillna(0) + early_exit_energy,
    # )
    
    # acc.append(oracle_acc)
    # energy_usage.append(optimal_cascade_exit_energy)
    # labels.append(f"Optimal cascade + exit {name}")

    # # Optimal reverse cascade at step 2, then early exit
    # early_exit_energy = np.where(
    #     data1["agent_output_eval"] == "CORRECT",
    #     data1["energy_total_mWh"],
    #     data1["step_0_energy_total_mWh"] + data1["step_1_energy_total_mWh"].fillna(0),
    # )
    # optimal_reverse_cascade_energy = np.where(
    #     (data2["agent_output_eval"] == "CORRECT"),
    #     data2["energy_total_mWh"],
    #     data2["step_0_energy_total_mWh"] + data2["step_1_energy_total_mWh"].fillna(0) + early_exit_energy,
    # )
    # acc.append(oracle_acc)
    # energy_usage.append(optimal_reverse_cascade_energy)
    # labels.append(f"Optimal reverse cascade + exit {name}")


In [ ]:
df1 = pd.read_csv("../data/frames/profile_results_frames_ollama_qwen3_fixed_cascade_compress_1.7b_to_30b_step_1_judged.csv")
df2 = pd.read_csv("../data/frames/profile_results_frames_ollama_qwen3_fixed_cascade_compress_1.7b_to_30b_step_2_judged.csv")

df2[(df2["agent_output_eval"] != "CORRECT") & (df1["agent_output_eval"] == "CORRECT")]["num_LLM_calls"]

In [ ]:
colors = sns.color_palette("husl", n_colors=len(labels))

plt.figure(figsize=(8, 5))
handles = []

for i in range(len(energy_usage)):
    bp = plt.boxplot(
        [energy_usage[i]],
        vert=False,
        positions=[acc[i]],
        widths=0.01,
        showfliers=False,
        patch_artist=True
    )

    # Only color the box body
    box = bp['boxes'][0]
    box.set_facecolor(colors[i])
    box.set_linewidth(1)  # thinner border
    # Customize median line
    median = bp['medians'][0]
    median.set_color('black')

    # Add to legend
    handles.append(Patch(facecolor=colors[i], label=labels[i]))

# Final formatting
ax = plt.gca()
ax.set_yticks([])  # wipe all old tick positions
ax.set_yticklabels([])  # wipe all old labels
yticks = np.arange(np.floor(min(acc) * 20) / 20, np.ceil(max(acc) * 20) / 20 + 0.001, 0.05)
ax.set_ylim(np.min(yticks) - 0.03, np.max(yticks) + 0.03)
ax.set_yticks(yticks)
ax.set_yticklabels([f"{y:.2f}" for y in yticks])

plt.grid(True, axis='x')
plt.xlabel("Energy Usage (mWh)")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Energy Usage on FRAMES")
plt.legend(handles=handles, title="Models", fontsize=8)
plt.tight_layout()
# plt.savefig("../figures/ollama_frames.png", dpi=300)
plt.show()

In [ ]:
colors = sns.color_palette("husl", n_colors=len(labels))

plt.figure(figsize=(8, 5))
handles = []

for i in range(len(energy_usage)):
    data = np.array(energy_usage[i])
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    n = len(data)
    ci95 = 1.96 * std / np.sqrt(n)

    lower = mean - ci95
    upper = mean + ci95
    y = acc[i]

    # Horizontal CI line
    plt.hlines(y, lower, upper, color=colors[i], lw=2)

    # Mean line (thicker)
    plt.vlines(mean, y - 0.005, y + 0.005, color=colors[i], lw=3)

    # CI end caps (minor vertical lines)
    plt.vlines(lower, y - 0.003, y + 0.003, color=colors[i], lw=1)
    plt.vlines(upper, y - 0.003, y + 0.003, color=colors[i], lw=1)

    handles.append(Patch(facecolor=colors[i], label=labels[i]))

# Final formatting
ax = plt.gca()
ax.set_yticks([])
ax.set_yticklabels([])
yticks = np.arange(np.floor(min(acc) * 20) / 20,
                   np.ceil(max(acc) * 20) / 20 + 0.001, 0.05)
ax.set_ylim(np.min(yticks) - 0.03, np.max(yticks) + 0.03)
ax.set_yticks(yticks)
ax.set_yticklabels([f"{y:.2f}" for y in yticks])

plt.grid(True, axis='x')
plt.xlabel("Energy Usage (mWh)")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Mean Energy Usage with 95% CI on FRAMES")
plt.legend(handles=handles, title="Models", loc="upper left", fontsize=6)
plt.tight_layout()
# plt.savefig("../figures/ollama_frames_mean_ci.png", dpi=300)
plt.show()

In [ ]:
data = pd.read_csv("../data/frames/profile_results_frames_full_llamacpp_qwen3_30b_judged.csv")

def plot_step_energy_stats():
    step_energy_data = []
    for step in range(1, 12):
        step_energy_data.append(data[(data["num_LLM_calls"] == step) & (data["agent_output_eval"] != "CORRECT")]["energy_total_mWh"])
        step_energy_data.append(data[(data["num_LLM_calls"] == step) & (data["agent_output_eval"] == "CORRECT")]["energy_total_mWh"])
    
    positions = []
    colors = []
    for i in range(1, 12):
        positions.extend([i-0.15, i+0.15])
        colors.extend(["red", "green"])
    
    plt.figure(figsize=(8, 5))
    for i in range(len(step_energy_data)):
        bp = plt.boxplot(
            [step_energy_data[i]],
            positions=[positions[i]],
            showfliers=False,
            patch_artist=True,
            widths=0.25
        )

        box = bp['boxes'][0]
        box.set_facecolor(colors[i])
        box.set_linewidth(1)  # thinner border
        # Customize median line
        median = bp['medians'][0]
        median.set_color('black')

    ax = plt.gca()
    ax.set_xticks([])
    ax.set_xticklabels([])
    xticks = np.arange(1, 12, 1)
    ax.set_xlim(0, 12)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{y}" for y in xticks])
    
    plt.xlabel("Number of LLM steps")
    plt.ylabel(f"Total energy usage (mWh)")
    plt.title(f"Total energy usage vs Step")
    plt.show()

plot_step_energy_stats()

In [ ]:
data1 = pd.read_csv("../data/frames/profile_results_frames_full_llamacpp_qwen3_1.7b_judged.csv")
data2 = pd.read_csv("../data/frames/profile_results_frames_full_llamacpp_qwen3_30b_judged.csv")

data1["energy_diff"] = data2["energy_total_mWh"] - data1["energy_total_mWh"]
data1["agent_output_eval_2"] = data2["agent_output_eval"]

plt.boxplot(
    [
        data1[(data1["agent_output_eval"] == "CORRECT") & (data2["agent_output_eval"] == "CORRECT")]["energy_diff"],
        data1[(data1["agent_output_eval"] == "CORRECT") & (data2["agent_output_eval"] != "CORRECT")]["energy_diff"],
        data1[(data1["agent_output_eval"] != "CORRECT") & (data2["agent_output_eval"] == "CORRECT")]["energy_diff"],
        data1[(data1["agent_output_eval"] != "CORRECT") & (data2["agent_output_eval"] != "CORRECT")]["energy_diff"],
    ],
    positions=[0, 1, 2, 3],
    # showfliers=False,
    patch_artist=True,
    widths=0.25
)
plt.xticks([0, 1, 2, 3], ["Both correct", "1.7B correct, 30B incorrect", "1.7B incorrect, 30B correct", "Both incorrect"], rotation=45)

# plt.hist(
#     data1[data1["agent_output_eval"] == "CORRECT"]["energy_diff"],
#     bins=10, alpha=0.7, label='1.7B correct', color="green", edgecolor='black', density=True
# )
# plt.hist(
#     data1[(data1["agent_output_eval"] != "CORRECT") & (data2["agent_output_eval"] == "CORRECT")]["energy_diff"],
#     bins=10, alpha=0.5, label='1.7B incorrect, 30B correct', color="red", edgecolor='black', density=True
# )
# plt.hist(
#     data1[(data1["agent_output_eval"] != "CORRECT") & (data2["agent_output_eval"] != "CORRECT")]["energy_diff"],
#     bins=10, alpha=0.5, label='1.7B incorrect, 30B incorrect', color="blue", edgecolor='black', density=True
# )

# plt.legend()

plt.show()